# Project 71 — Fine-Tuning Dataset Builder
## Generate Instruction–Response Training Pairs

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Define Domain & Seed Examples

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd, json

llm = ChatOllama(model="qwen3:8b", temperature=0.7)

domain = "customer support for a SaaS product"

seed_examples = [
    {"instruction": "How do I reset my password?",
     "response": "Go to Settings > Security > Reset Password. You'll receive a confirmation email."},
    {"instruction": "Can I upgrade my plan mid-billing cycle?",
     "response": "Yes! Go to Billing > Change Plan. You'll be prorated for the remaining days."},
    {"instruction": "Why is my dashboard loading slowly?",
     "response": "Try clearing your browser cache. If it persists, check our status page for outages."},
]
print(f"Domain: {domain}")
print(f"Seed examples: {len(seed_examples)}")

## Step 2 — Generate New Training Pairs

In [ ]:
gen_prompt = ChatPromptTemplate.from_messages([
    ("system", f"""You are generating training data for a {domain} chatbot.

Given seed examples, generate 5 NEW diverse instruction-response pairs.
Cover different topics: billing, technical issues, features, account management.

Return a JSON array of objects with "instruction" and "response" keys.
Return ONLY valid JSON, no extra text."""),
    ("human", "Seed examples:\n{seeds}\n\nGenerate 5 new pairs:")
])
gen_chain = gen_prompt | llm | StrOutputParser()

seeds_text = json.dumps(seed_examples, indent=2)
raw = gen_chain.invoke({"seeds": seeds_text})

# Parse generated pairs
start = raw.find("[")
end = raw.rfind("]") + 1
if start >= 0 and end > start:
    generated = json.loads(raw[start:end])
else:
    generated = []
    print("Warning: Could not parse JSON, using fallback")

all_pairs = seed_examples + generated
print(f"Generated {len(generated)} new pairs (total: {len(all_pairs)})")
for pair in generated[:3]:
    print(f"  Q: {pair['instruction'][:60]}")
    print(f"  A: {pair['response'][:60]}...")
    print()

## Step 3 — Quality Check & Format

In [ ]:
quality_prompt = ChatPromptTemplate.from_messages([
    ("system", """Rate this training pair on a 1-5 scale for:
- relevance: Is it on-topic for customer support?
- clarity: Is the response clear and helpful?
- correctness: Is the response accurate?

Return JSON: {{"relevance": N, "clarity": N, "correctness": N, "issues": "..."}}"""),
    ("human", "Instruction: {instruction}\nResponse: {response}")
])
quality_chain = quality_prompt | llm | StrOutputParser()

checked_pairs = []
for pair in all_pairs:
    try:
        raw = quality_chain.invoke(pair)
        s = raw.find("{"); e = raw.rfind("}") + 1
        scores = json.loads(raw[s:e]) if s >= 0 else {"relevance":3,"clarity":3,"correctness":3}
    except Exception:
        scores = {"relevance": 3, "clarity": 3, "correctness": 3}

    pair["quality_score"] = (scores.get("relevance",3) + scores.get("clarity",3) + scores.get("correctness",3)) / 3
    checked_pairs.append(pair)

df = pd.DataFrame(checked_pairs)
print(f"Quality scores:")
print(f"  Mean: {df['quality_score'].mean():.2f}")
print(f"  Min:  {df['quality_score'].min():.2f}")
print(f"  Max:  {df['quality_score'].max():.2f}")

## Step 4 — Export in Multiple Formats

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

# Alpaca format
alpaca = [{"instruction": p["instruction"], "input": "", "output": p["response"]} for p in checked_pairs]
with open("sample_data/training_alpaca.json", "w") as f:
    json.dump(alpaca, f, indent=2)

# ShareGPT format
sharegpt = [{"conversations": [
    {"from": "human", "value": p["instruction"]},
    {"from": "gpt", "value": p["response"]}
]} for p in checked_pairs]
with open("sample_data/training_sharegpt.json", "w") as f:
    json.dump(sharegpt, f, indent=2)

# JSONL format
with open("sample_data/training.jsonl", "w") as f:
    for p in checked_pairs:
        json.dump({"prompt": p["instruction"], "completion": p["response"]}, f)
        f.write("\n")

print(f"Exported {len(checked_pairs)} pairs in 3 formats:")
print("  ✓ sample_data/training_alpaca.json")
print("  ✓ sample_data/training_sharegpt.json")
print("  ✓ sample_data/training.jsonl")

## What You Learned
- **Seed-based data augmentation** for training pairs
- **Quality scoring** with LLM-as-judge
- **Export formats:** Alpaca, ShareGPT, JSONL